# Python 函数进阶 + 异常处理 + 文件 IO（习题）

Easy

1. 写函数 power(base, exp=2),求 base 的 exp 次方,exp 默认 2。测试:power(3)、power(2, 10)。

In [78]:
def power(base:float, exp:int=2) -> float:
    return float(base**exp)

print(power(3))
print(power(2, 10))

9.0
1024.0


2. 写函数 my_max(*args),接收任意多个数字,返回最大值。不许用内置 max()(自己实现)。想想:如果一个参数都不传,该怎么办?

In [12]:
def my_max(*args: float) -> float | None:
    maximum = float('-inf')
    if args:
        for arg in args:
            if arg > maximum:
                maximum = arg
        return maximum
    else:
        print("没有传递参数")
        return None
        

print(my_max(3, 1, 4, 1, 5))

5


In [13]:
print(my_max())

没有传递参数
None


参考答案

float('-inf') 初始化——你弱点 #18 学的东西用上了,空输入也处理了。漂亮。

一个风格建议:你的结构是 if args: ...return... else: ...return None。可以用「卫语句(guard clause)」更扁平:

In [79]:
def my_max(*args: float) -> float | None:
    if not args:
        print("没有传递参数")
        return None
    maximum = float('-inf')
    for arg in args:
        if arg > maximum:
            maximum = arg
    return maximum

先处理异常情况、提前 return,主逻辑不缩进。读起来「异常 → 滚开;下面都是正常路径」。你 Day 3 学的可读性原则,卫语句是其中很实用的一条。

3. 写函数 safe_int(s):把字符串转成整数,转不了就返回 None(不许让程序崩)。测试 "42"、"abc"、""、None。

In [29]:
def safe_int(s:str) -> int | None:
    try:
        return int(s)
    except (ValueError, TypeError):
        return None
    
print(safe_int("42"))

42


In [30]:
print(safe_int("abc"))

None


In [31]:
print(safe_int(""))

None


In [32]:
print(safe_int(None))

None


Medium

4. 写函数 make_order(product, quantity=1, **kwargs):返回一个 dict,包含 product、quantity,以及 kwargs 里所有额外字段。测试时传一些额外字段(如 discount=0.1)。

In [ ]:
def make_order(product:str, quantity:int=1, **kwargs) -> dict:
    order = {
        "product":product,
        "quantity":quantity
    }
    order.update(kwargs)
    return order

In [37]:
print(make_order("banana",20,discount=0.1))

{'product': 'banana', 'quantity': 20, 'discount': 0.1}


5. 写函数 read_numbers(filename):读一个每行一个数字的文本文件,返回数字列表。要求:文件不存在时不崩溃,返回空列表 [];文件里某行不是数字时跳过那行。
(先自己用 with open(... 'w') 造一个测试文件,故意混进一行 abc。)

In [46]:
with open("nums_file.txt", 'w', encoding='utf-8') as f:
    f.write("123\n")
    f.write("234\n")
    f.write("abc\n")
    f.write("345\n")


In [47]:
def read_numbers(filename:str) -> list:
    num_lst = []
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines:
            try:
                num_lst.append(float(line))
            except ValueError:
                continue
    return num_lst

In [48]:
read_numbers("nums_file.txt")

[123.0, 234.0, 345.0]

参考答案

代码逻辑对,结果 [123.0, 234.0, 345.0] 正确,abc 行被跳过。但题目明确要求「文件不存在时不崩溃,返回空列表 []」——你没处理这个。

现在如果调 read_numbers("不存在.txt"),会直接抛 FileNotFoundError 崩掉。验证一下你就知道了:

In [80]:
def read_numbers(filename: str) -> list:
    num_lst = []
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:                    # 直接迭代 f,不用 readlines()
                try:
                    num_lst.append(float(line))
                except ValueError:
                    continue
    except FileNotFoundError:
        return []
    return num_lst

两个改进:

FileNotFoundError 没处理 —— 这是题目要求,属于漏做。这也是你弱点 #19 的精神:题目都明说了要测「文件不存在」,你却没造这个用例。

f.readlines() → 直接 for line in f:readlines() 把整个文件一次性读进内存成一个 list。文件大就爆内存。for line in f 是逐行流式读取,文件多大都不怕。这是处理文件的标准姿势,记进笔记。

6. 给题 1 的 power 函数加类型注解(参数和返回值)。再写一个函数 divide(a: float, b: float) -> float | None,除数为 0 时返回 None。

In [52]:
def divide(a:float, b:float) -> float | None:
    try:
        return a / b 
    except ZeroDivisionError:
        return None

In [53]:
print(divide(2,3))

0.6666666666666666


In [54]:
print(divide(2,0))

None


7. 写一个装饰器 log_call:被它装饰的函数,每次调用前打印 "调用 函数名,参数 ...",调用后打印 "返回 ..."。用它装饰题 1 的 power,验证效果。

In [82]:
def log_call(func):
    def wrapper(*args):
        print(f"调用{func.__name__}，参数{args}")
        result = func(*args)
        print(f"返回{result}")
    return wrapper

@log_call
def power(base:float, exp:int=2) -> float:
    return base**exp

power(3,2)

调用power，参数(3, 2)
返回9


参考答案

题 7 — ⚠️ 真 bug:wrapper 吞掉了返回值 + 漏接 kwargs

Bug 1:wrapper 没把 result return 出去。 你装饰后的 power 调用完,打印是对的,但函数返回值变成了 None。

In [84]:
@log_call
def power(base:float, exp:int=2) -> float:
    return base**exp
x = power(3, 2)
print(x)        # None ！原函数明明该返回 9     

调用power，参数(3, 2)
返回9
None


装饰器 timer、log_call 这类「只是加旁路功能」的,必须把原函数的返回值原样 return 出去,否则等于偷偷改坏了原函数。这是装饰器最经典的 bug。跟练 Step 4 的 timer 里有 return result,你照着写时漏了。

Bug 2:def wrapper(*args) 漏了 **kwargs。 现在如果有人 power(base=3, exp=2) 用关键字传参,wrapper 接不住会报错。装饰器的 wrapper 永远写 (*args, **kwargs)——你题 9 的 retry 就写对了,题 7 反而漏了。

正确版:

In [85]:
def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"调用 {func.__name__},参数 args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"返回 {result}")
        return result          # ← 关键
    return wrapper

铁律:写装饰器 wrapper,两件事不能忘——① 签名写 (*args, **kwargs) ② 把 func(...) 的结果 return 出去。这条进弱点清单。

8. 写函数 summarize(filename):读取 data/sales.csv,不用 Pandas,用 with open + csv 模块,统计并返回一个 dict:{总行数, 总销售额, 平均订单金额}。提示:import csv,csv.DictReader 能把每行读成 dict。
(这题是「手写一遍 Pandas 干的事」——理解底层。)

In [63]:
import csv
def summarize(filename:str) -> dict:
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        total_rows = 0
        total_sales = 0
        for row in reader:
            total_rows += 1
            total_sales += float(row.get('total', 0))
        avg_order_sales = total_sales/total_rows

    return {
        "总行数": total_rows,
        "总销售额": round(total_sales, 2),
        "平均订单金额": round(avg_order_sales, 2)
    }


In [64]:
summarize("../data/sales.csv")

{'总行数': 500, '总销售额': 1267716.0, '平均订单金额': 2535.43}

参考答案

逻辑对,csv.DictReader 用法正确,结果 {总行数:500, 总销售额:1267716.0, 平均订单金额:2535.43} 准确。

一个隐藏 bug 苗头:avg_order_sales = total_sales / total_rows。如果文件是空的(只有表头、0 行数据),total_rows = 0,这里 除以 0 直接崩 ZeroDivisionError。

你弱点 #19 的精神:造个空 CSV 测一下就暴露了。修法:

In [86]:
# avg_order_sales = total_sales / total_rows if total_rows else 0

9. 写一个装饰器 retry:被装饰的函数如果抛异常,自动重试最多 3 次,3 次都失败才真正抛出异常。每次重试打印第几次。
测试:写一个「前两次抛异常、第三次成功」的函数验证。(提示:用一个外部计数器,或函数内部状态。)

In [71]:
def retry(func):
    def wrapper(*args, **kwargs):
        max_retry = 3
        for i in range(1, max_retry+1):
            try:
                return(func(*args, **kwargs))
            except Exception as e:
                if i == max_retry:
                    raise e
                print(f"第{i}次重试，函数执行失败，原因：{str(e)}")
    return wrapper

@retry
def test_simpler(call_count=[0]): 
    call_count[0] += 1
    if call_count[0] < 3:
        raise Exception(f"第{call_count[0]}次调用出错")
    return "第3次调用成功！"
test_simpler()

第1次重试，函数执行失败，原因：第1次调用出错
第2次重试，函数执行失败，原因：第2次调用出错


'第3次调用成功！'

10. —(综合 + 刁钻测试) 写函数 process_sales(records):接收一个订单 dict 列表,返回按 country 分组的总销售额 dict。要求:

• 用 try/except 跳过格式有问题的记录(缺字段、total 不是数字)

• 自己构造测试数据:故意混入一条缺 country 的、一条 total 是字符串 "abc" 的、一条正常的

• (b) 在 cell 里写注释:你的函数遇到坏数据时的行为是「跳过」还是「崩溃」?为什么跳过比崩溃好?

In [ ]:
def process_sales(records: list[dict]) -> dict:
    total_country_sales = {}

    for record in records:
        try:
            country = record.get("country")
            total = float(record.get("total", 0))
            if country in total_country_sales:
                total_country_sales[country] += total
            else:
                total_country_sales[country] = total
        except (KeyError, ValueError, TypeError) as e:
            continue
            print(f"数据格式错误: {e}")
    
    return total_country_sales


In [76]:
test_orders = [
    # 正常记录1
    {"country": "中国", "total": 100, "order_id": "001"},
    # 正常记录2
    {"country": "美国", "total": 200, "order_id": "002"},
    # 坏数据1：缺 country 字段
    {"total": 150, "order_id": "003"},
    # 坏数据2：total 是字符串 "abc"，无法转为数字
    {"country": "日本", "total": "abc", "order_id": "004"},
    # 正常记录3：同国家，验证累加逻辑
    {"country": "中国", "total": 50, "order_id": "005"},
    # 正常记录4：浮点数销售额，验证数值转换
    {"country": "德国", "total": 89.9, "order_id": "006"},
]

In [77]:
process_sales(test_orders)

数据格式错误: could not convert string to float: 'abc'


{'中国': 150.0, '美国': 200.0, None: 150.0, '德国': 89.9}

我认为跳过好，销售数据统计是典型的批量数据处理场景，单条订单的格式错误，不应该影响整个统计任务的完成。如果直接崩溃，会导致所有有效数据的统计结果都无法输出，完全无法完成核心业务目标；而跳过坏数据，至少能输出所有有效数据的统计结果，保证核心任务完成。

参考答案

看你的结果:{'中国': 150.0, '美国': 200.0, None: 150.0, '德国': 89.9}。

那条「缺 country」的坏数据没被跳过,反而以 None 当 key 混进了结果。 题目明确要求「跳过缺字段的记录」,你的结果里有 None: 150.0——这就是没跳成。

为什么没跳成?看你的代码:

In [87]:
# country = record.get("country")    # 缺字段时,.get() 返回 None,不报错!
# total = float(record.get("total", 0))

你预期靠 except KeyError 来抓「缺字段」。但 .get() 永远不会抛 KeyError——它缺 key 时安静地返回 None。所以 country 缺失这条根本没进 except,None 顺利变成了 dict 的 key。

这是个很值得记的点:record["country"] 缺 key 会抛 KeyError;record.get("country") 缺 key 返回 None 不抛错。 你想靠 except 抓缺字段,就不能用 .get()。这呼应你 Day 2 笔记「.get() 安全访问避免 KeyError」——但「安全」是把双刃剑,这里你需要它报错,它却不报。

修法二选一:

In [88]:
# 方法 A:用 [] 取值,缺 key 真的抛 KeyError,被 except 抓到
# for record in records:
#     try:
#         country = record["country"]      # 缺 → KeyError
#         total = float(record["total"])   # 缺 → KeyError;非数字 → ValueError
#         total_country_sales[country] = total_country_sales.get(country, 0) + total
#     except (KeyError, ValueError, TypeError):
#         continue

In [ ]:
# 方法 B:继续用 .get(),但显式检查 None,主动跳过
# for record in records:
#     country = record.get("country")
#     if country is None:          # 显式判断
#         continue
#     try:
#         total = float(record.get("total", 0))
#     except (ValueError, TypeError):
#         continue
#     total_country_sales[country] = total_country_sales.get(country, 0) + total

还有一个「死代码」bug:

In [ ]:
# except (KeyError, ValueError, TypeError) as e:
#     continue
#     print(f"数据格式错误: {e}")    # ← 这行永远执行不到!

continue 之后的 print 永远不会执行——continue 一旦执行,本次循环立刻跳到下一轮,后面的代码是死代码(dead code)。

奇怪的是你的输出里确实打印了「数据格式错误: could not convert...」——那是 total="abc" 那条,它进 except 时……等等,按代码逻辑这行不该执行。说明你实际运行的代码和上传的不是同一版(可能跑的时候 print 在 continue 前面,后来改了顺序又没重跑)。

不管怎样,记住:continue/return/break/raise 之后的同层代码都是死代码。 想既打印又跳过,把 print 放 continue 前面:

In [ ]:
# except (KeyError, ValueError, TypeError) as e:
#     print(f"数据格式错误: {e}")
#     continue